In [26]:
print("kernel")

kernel


In [ ]:
import pandas as pd
import numpy as np

# ML
from sklearn.model_selection import train_test_split
#from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

# Preprocessing
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Metrics
from sklearn.metrics import classification_report, roc_auc_score

# Imbalance
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline


# MLflow
import mlflow
import mlflow.sklearn

import warnings
warnings.filterwarnings("ignore")

RANDOM_STATE = 42
TEST_SIZE = 0.2

mlflow.set_tracking_uri("file:///mnt/c/Users/under/Documents/F5/3_Projects/Stroker_project/mlruns")
mlflow.set_experiment("stroke_project")

2026/04/16 22:44:19 INFO mlflow.tracking.fluent: Experiment with name 'stroke_project' does not exist. Creating a new experiment.


<Experiment: artifact_location='file:///mnt/c/Users/under/Documents/F5/3_Projects/Stroker_project/mlruns/671783000064998482', creation_time=1776372259186, experiment_id='671783000064998482', last_update_time=1776372259186, lifecycle_stage='active', name='stroke_project', tags={}, trace_location=None, workspace='default'>

In [28]:
df = pd.read_csv("../data/raw/stroke_dataset.csv")

In [29]:
def get_dataset(version="full"):
    
    df_copy = df.copy()
    
    # Limpieza básica basada en EDA
    df_copy.loc[df_copy["work_type"] == "children", "work_type"] = "not_applied"
    
    if version == "adults":
        df_copy = df_copy[df_copy["age"] >= 17]
    
    return df_copy

In [30]:
def build_preprocessor(X):
    
    cat_cols = X.select_dtypes(include="object").columns
    num_cols = X.select_dtypes(exclude="object").columns
    
    preprocessor = ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", StandardScaler(), num_cols)
    ])
    
    return preprocessor

In [ ]:

def run_experiment(model, model_name, dataset_version="full", use_smote=False):
    
    df_exp = get_dataset(dataset_version)
    
    X = df_exp.drop("stroke", axis=1)
    y = df_exp["stroke"]
    
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        stratify=y,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE
    )
    
    preprocessor = build_preprocessor(X_train)
    
    # Pipeline dinámico
    steps = [("prep", preprocessor)]

    if use_smote:
        steps.append(("smote", SMOTE(random_state=42)))

    steps.append(("model", model))

    pipeline = Pipeline(steps)
    
    # MLFlow
    with mlflow.start_run():
        
        pipeline.fit(X_train, y_train)
        
        y_pred = pipeline.predict(X_test)
        y_prob = pipeline.predict_proba(X_test)[:, 1]
        
        auc = roc_auc_score(y_test, y_prob)
        
        print(f"\n=== {model_name} | {dataset_version} | SMOTE={use_smote} ===")
        print(classification_report(y_test, y_pred))
        print("AUC:", auc)
        
        mlflow.log_param("model", model_name)
        mlflow.log_param("dataset", dataset_version)
        mlflow.log_param("smote", use_smote)
        mlflow.log_metric("auc", auc)
        
        mlflow.sklearn.log_model(pipeline, model_name)

In [32]:
log_model = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    random_state=RANDOM_STATE
)

rf_model = RandomForestClassifier(
    n_estimators=100,
    class_weight="balanced",
    random_state=RANDOM_STATE
)

In [33]:
run_experiment(log_model, "Logistic", "full", use_smote=False)
run_experiment(rf_model, "RandomForest", "full", use_smote=False)


=== Logistic | full | SMOTE=False ===
              precision    recall  f1-score   support

           0       0.98      0.75      0.85       947
           1       0.14      0.76      0.23        50

    accuracy                           0.75       997
   macro avg       0.56      0.75      0.54       997
weighted avg       0.94      0.75      0.82       997

AUC: 0.8372756071805703


2026/04/16 22:44:20 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/16 22:44:22 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



=== RandomForest | full | SMOTE=False ===
              precision    recall  f1-score   support

           0       0.95      0.96      0.96       947
           1       0.12      0.10      0.11        50

    accuracy                           0.92       997
   macro avg       0.54      0.53      0.53       997
weighted avg       0.91      0.92      0.91       997

AUC: 0.8130623020063359


2026/04/16 22:44:29 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/16 22:44:30 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


In [34]:
run_experiment(log_model, "Logistic", "full", use_smote=True)
run_experiment(rf_model, "RandomForest", "full", use_smote=True)


=== Logistic | full | SMOTE=True ===
              precision    recall  f1-score   support

           0       0.98      0.75      0.85       947
           1       0.14      0.76      0.23        50

    accuracy                           0.75       997
   macro avg       0.56      0.75      0.54       997
weighted avg       0.94      0.75      0.82       997

AUC: 0.8372756071805703


2026/04/16 22:44:34 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/16 22:44:35 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



=== RandomForest | full | SMOTE=True ===
              precision    recall  f1-score   support

           0       0.95      0.96      0.96       947
           1       0.12      0.10      0.11        50

    accuracy                           0.92       997
   macro avg       0.54      0.53      0.53       997
weighted avg       0.91      0.92      0.91       997

AUC: 0.8130623020063359


2026/04/16 22:44:41 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/16 22:44:43 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


In [35]:
run_experiment(log_model, "Logistic", "adults", use_smote=False)
run_experiment(rf_model, "RandomForest", "adults", use_smote=True)


=== Logistic | adults | SMOTE=False ===
              precision    recall  f1-score   support

           0       0.98      0.70      0.82       794
           1       0.14      0.80      0.24        49

    accuracy                           0.71       843
   macro avg       0.56      0.75      0.53       843
weighted avg       0.93      0.71      0.79       843

AUC: 0.838893743895543


2026/04/16 22:44:48 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/16 22:44:50 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



=== RandomForest | adults | SMOTE=True ===
              precision    recall  f1-score   support

           0       0.95      0.97      0.96       794
           1       0.21      0.14      0.17        49

    accuracy                           0.92       843
   macro avg       0.58      0.56      0.56       843
weighted avg       0.91      0.92      0.91       843

AUC: 0.77687246183108


2026/04/16 22:44:57 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/16 22:44:58 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
